# Million-Plus Cities: Labour Market + Enterprise Structure
## End-to-end Pandas notebook from the original PDFs

This notebook deliberately starts from the **original source PDFs** rather than a pre-cleaned dataset.

The full workflow is:

**PDFs → inspection → PDF-to-text → table parsing → source-table CSVs → cleaning → quality checks → PLFS/ASUSE merge → derived variables → analysis tables → selected charts → final exports**

### Primary sources

1. **Labour Market Dynamics in Million-plus Cities** — PLFS 2025 city-level report.
2. **Urban Unincorporated Enterprise Landscape: ASUSE 2025 – Insights from Million-plus Cities** — enterprise-side city indicators.
3. **PLFS Monthly Bulletin, July 2026** — used only for the latest urban labour-market pulse.

### Editorial question

> **How different are India's million-plus cities in labour utilisation, employment structure and the productivity of their unincorporated business economy?**

The analysis is descriptive. PLFS measures people/labour-market status; ASUSE measures establishments/workers in the covered unincorporated non-agricultural sector. The indicators have different concepts and denominators.

## 0. Reproducibility setup

The notebook creates separate folders for raw extracted text, parsed CSVs, charts and final outputs.

When run in the supplied environment it will automatically find the attached PDFs in `/mnt/data`. When run elsewhere, place the three PDFs in `source_pdfs/` beside the notebook or change `SOURCE_DIR`.

In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
ATTACHED_DIR = Path("/mnt/data")

if (ATTACHED_DIR / "Labour Market Dynamics in Million-plus Cities(1).pdf").exists():
    SOURCE_DIR = ATTACHED_DIR
else:
    SOURCE_DIR = PROJECT_DIR / "source_pdfs"

RAW_TEXT_DIR = PROJECT_DIR / "raw_text"
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHART_DIR = PROJECT_DIR / "charts"
WORK_DIR = PROJECT_DIR / "work"

for folder in [RAW_TEXT_DIR, OUTPUT_DIR, CHART_DIR, WORK_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Source :", SOURCE_DIR)
print("Python :", sys.version.split()[0])

## 1. Locate the original input files

This cell makes the exact input filenames visible before any transformation occurs.

In [ ]:
def choose_existing(candidates):
    for filename in candidates:
        path = SOURCE_DIR / filename
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find any of:\n" + "\n".join("  " + str(SOURCE_DIR / x) for x in candidates)
    )

SOURCE_FILES = {
    "labour_city": choose_existing([
        "Labour Market Dynamics in Million-plus Cities(1).pdf",
        "Labour Market Dynamics in Million-plus Cities.pdf",
    ]),
    "asuse_city": choose_existing([
        "ASUSE Insights in Million-pus Cities.pdf",
    ]),
    "plfs_july": choose_existing([
        "PLFS July(1).pdf",
        "PLFS July.pdf",
    ]),
}

source_inventory = pd.DataFrame([
    {
        "source_key": key,
        "file": path.name,
        "size_kb": round(path.stat().st_size / 1024, 1),
        "path": str(path),
    }
    for key, path in SOURCE_FILES.items()
])

display(source_inventory)

## 2. Inspect the PDFs

Before extraction, check page counts and basic PDF metadata.

In [ ]:
def pdf_info(path):
    result = subprocess.run(
        ["pdfinfo", str(path)],
        capture_output=True,
        text=True,
        check=True,
    )
    info = {}
    for line in result.stdout.splitlines():
        if ":" in line:
            key, value = line.split(":", 1)
            info[key.strip()] = value.strip()
    return info

pdf_inventory = pd.DataFrame([
    {
        "source_key": key,
        "file": path.name,
        "pages": pdf_info(path).get("Pages"),
        "page_size": pdf_info(path).get("Page size"),
        "pdf_version": pdf_info(path).get("PDF version"),
    }
    for key, path in SOURCE_FILES.items()
])

display(pdf_inventory)

## 3. Optional visual QA of a source page

PDF table extraction should be checked against the original page layout. This optional cell renders one source page as a PNG using Poppler. It is not used to parse the numbers; it is a visual sanity check.

In [ ]:
# Optional: render a representative page from the city-level PLFS report.
# Change PAGE_NUMBER as needed when inspecting other tables.
PAGE_NUMBER = 27  # 1-based PDF page number

preview_prefix = WORK_DIR / "labour_page_preview"
render_result = subprocess.run(
    [
        "pdftoppm",
        "-f", str(PAGE_NUMBER),
        "-l", str(PAGE_NUMBER),
        "-png",
        "-singlefile",
        str(SOURCE_FILES["labour_city"]),
        str(preview_prefix),
    ],
    capture_output=True,
    text=True,
)

preview_png = Path(str(preview_prefix) + ".png")
if render_result.returncode == 0 and preview_png.exists():
    from IPython.display import Image
    display(Image(filename=str(preview_png), width=900))
else:
    print("Visual render skipped:", render_result.stderr.strip())

## 4. Convert PDF to layout-preserving text

This is the actual **PDF → text** conversion step.

`pdftotext -layout` keeps the horizontal positioning of table columns as far as possible, which makes the later parser substantially safer than trying to treat the PDF as a normal text document.

In [ ]:
def pdf_to_layout_text(pdf_path, text_path):
    subprocess.run(
        ["pdftotext", "-layout", str(pdf_path), str(text_path)],
        check=True
    )
    return text_path

TEXT_FILES = {
    "labour_city": pdf_to_layout_text(
        SOURCE_FILES["labour_city"],
        RAW_TEXT_DIR / "labour_market_dynamics_2025.txt"
    ),
    "asuse_city": pdf_to_layout_text(
        SOURCE_FILES["asuse_city"],
        RAW_TEXT_DIR / "asuse_2025_million_plus_cities.txt"
    ),
    "plfs_july": pdf_to_layout_text(
        SOURCE_FILES["plfs_july"],
        RAW_TEXT_DIR / "plfs_july_2026.txt"
    ),
}

text_inventory = pd.DataFrame([
    {
        "source_key": key,
        "text_file": path.name,
        "size_kb": round(path.stat().st_size / 1024, 1),
    }
    for key, path in TEXT_FILES.items()
])

display(text_inventory)

## 5. Inspect the raw extracted text

Always inspect a small amount of the converted text before writing parsing rules.

In [ ]:
def read_text(path):
    return path.read_text(encoding="utf-8", errors="replace")

labour_txt = read_text(TEXT_FILES["labour_city"])
asuse_txt = read_text(TEXT_FILES["asuse_city"])
july_txt = read_text(TEXT_FILES["plfs_july"])

print("First 35 lines of the PLFS city report text:\n")
print("\n".join(labour_txt.splitlines()[:35]))

print("\n\nRelevant PLFS table headings found:")
for n, line in enumerate(labour_txt.splitlines(), start=1):
    if re.match(r"^Table (2:|2\.1:|3:|3\.1|4:|7:|8:|10:|11:|11\.1:|12:|13\.1:|13\.2:)", line):
        print(n, line.strip())

print("\nRelevant ASUSE headings found:")
for n, line in enumerate(asuse_txt.splitlines(), start=1):
    if line.startswith("Table 1:") or line.startswith("Table 2:") or line.strip().startswith("Annexure-IB"):
        print(n, line.strip())

print("\nPLFS monthly statement headings found:")
for n, line in enumerate(july_txt.splitlines(), start=1):
    if line.strip().startswith("Statement "):
        print(n, line.strip())

## 6. Parsing helpers

The parser keeps the raw token and the cleaned numeric value. This is useful for auditing `-` and `*` markers instead of silently discarding them.

In [ ]:
NUM_TOKEN_RE = re.compile(
    r"(?:-|(?:\d{1,3}(?:,\d{2})+,\d{3}|\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?\*?)"
)

def token_to_number(token):
    if token is None:
        return np.nan
    token = str(token).strip()
    if token == "-":
        return np.nan
    return float(token.replace(",", "").replace("*", ""))

def token_has_star(token):
    return isinstance(token, str) and token.strip().endswith("*")

def extract_nth_table(text, start_pattern, occurrence=0, stop_patterns=None):
    lines = text.splitlines()
    start_re = re.compile(start_pattern)

    starts = [
        i for i, line in enumerate(lines)
        if start_re.search(line)
    ]

    if occurrence >= len(starts):
        raise ValueError(
            f"Pattern {start_pattern!r} has {len(starts)} occurrence(s); "
            f"requested occurrence={occurrence}."
        )

    start = starts[occurrence]
    end = len(lines)

    if stop_patterns:
        stop_res = [re.compile(p) for p in stop_patterns]
        for i in range(start + 1, len(lines)):
            if any(rx.search(lines[i]) for rx in stop_res):
                end = i
                break

    return "\n".join(lines[start:end])

def clean_city_name(name):
    s = re.sub(r"\s+", " ", str(name).strip())
    s = re.sub(r"\s+MC$", "", s)
    s = re.sub(r"\s+Municipal Corporation$", "", s)
    return s

def parse_numeric_city_rows(block, n_values, valid_cities=None, continuation_names=None, city_regex=None):
    """
    Parse city rows from a layout-preserving table block.

    The reports do not all use the same header layout: PLFS has explicit (1)-(n)
    header rows while ASUSE table headers are text-only. Therefore the parser does
    not depend on a header marker. Instead it identifies rows by numeric-token count
    and, whenever available, by the known 46-city reference set.
    """
    continuation_names = set(continuation_names or [])
    rows = []
    pending_name = None
    city_rx = re.compile(city_regex) if city_regex else None

    for raw_line in block.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        matches = list(NUM_TOKEN_RE.finditer(line))
        if len(matches) < n_values:
            # ASUSE has one wrapped city name: "Chhatrapati" / "Sambhajinagar".
            if line in continuation_names:
                pending_name = line
            continue

        value_matches = matches[-n_values:]
        name = line[:value_matches[0].start()].strip()
        raw_values = [m.group(0) for m in value_matches]

        if pending_name:
            name = f"{pending_name} {name}".strip()
            pending_name = None

        name = re.sub(r"\s+", " ", name).strip()
        if not name:
            continue

        if valid_cities is not None and name not in valid_cities:
            continue
        if city_rx is not None and not city_rx.search(name):
            continue

        row = {"city_raw": name}
        for j, tok in enumerate(raw_values, start=1):
            row[f"value_{j}_raw"] = tok
            row[f"value_{j}"] = token_to_number(tok)
            row[f"value_{j}_rse_star"] = token_has_star(tok)
        rows.append(row)

    return pd.DataFrame(rows)

# PLFS city-level extraction

## 7. Parse PLFS Table 1: sample and the 46-city reference list

This table provides the city universe and sample size. We use it to create the canonical 46-city reference set for all subsequent PLFS parsing.

In [ ]:
plfs_table1_block = extract_nth_table(
    labour_txt,
    r"^Table 1: Sample FSU, Household and Persons",
    occurrence=0,
    stop_patterns=[r"^Table 2:"]
)

plfs_sample_raw = parse_numeric_city_rows(
    plfs_table1_block,
    n_values=5,
    city_regex=r"MC$",
)

plfs_sample_raw["city"] = plfs_sample_raw["city_raw"].map(clean_city_name)

plfs_sample = plfs_sample_raw[
    ~plfs_sample_raw["city"].isin(["all million-plus cities", "urban India"])
].copy()

# Table 1 columns are FSU, household, male, female and person counts.
# Keep the person count explicitly named because it is used downstream.
plfs_sample = plfs_sample.rename(columns={
    "value_1": "fsu_surveyed",
    "value_2": "households_surveyed",
    "value_3": "sample_male_15plus",
    "value_4": "sample_female_15plus",
    "value_5": "persons_surveyed",
})

plfs_cities_raw = set(plfs_sample_raw["city_raw"])
plfs_cities = set(plfs_sample["city"])

print("PLFS cities parsed:", len(plfs_cities))
display(plfs_sample.head(10))

## 8. Parse PLFS LFPR, WPR and unemployment rate

Tables 2, 2.1, 3, 3.1, 11 and 11.1 each have three numeric columns: male, female, person.

We retain the `*` flag alongside the numeric estimate.

In [ ]:
def parse_plfs_three_gender_table(table_start, next_table, prefix, occurrence=0):
    block = extract_nth_table(
        labour_txt,
        table_start,
        occurrence=occurrence,
        stop_patterns=[next_table],
    )

    raw = parse_numeric_city_rows(
        block,
        n_values=3,
        valid_cities=plfs_cities_raw,
    )
    raw["city"] = raw["city_raw"].map(clean_city_name)

    return raw[
        [
            "city",
            "value_1", "value_2", "value_3",
            "value_1_rse_star", "value_2_rse_star", "value_3_rse_star",
        ]
    ].rename(columns={
        "value_1": f"{prefix}_male",
        "value_2": f"{prefix}_female",
        "value_3": f"{prefix}_person",
        "value_1_rse_star": f"{prefix}_male_rse_star",
        "value_2_rse_star": f"{prefix}_female_rse_star",
        "value_3_rse_star": f"{prefix}_person_rse_star",
    })

plfs_lfpr_usual = parse_plfs_three_gender_table(
    r"^Table 2:", r"^Table 2\.1:", "lfpr_usual"
)
plfs_lfpr_cws = parse_plfs_three_gender_table(
    r"^Table 2\.1:", r"^Table 2\.2", "lfpr_cws"
)
plfs_wpr_usual = parse_plfs_three_gender_table(
    r"^Table 3:", r"^Table 3\.1", "wpr_usual"
)
plfs_wpr_cws = parse_plfs_three_gender_table(
    r"^Table 3\.1:", r"^Table 4:", "wpr_cws"
)
plfs_ur_usual = parse_plfs_three_gender_table(
    r"^Table 11:", r"^Table 11\.1:", "ur_usual"
)
plfs_ur_cws = parse_plfs_three_gender_table(
    r"^Table 11\.1:", r"^Table 12:", "ur_cws"
)

for label, df in {
    "LFPR usual": plfs_lfpr_usual,
    "LFPR CWS": plfs_lfpr_cws,
    "WPR usual": plfs_wpr_usual,
    "WPR CWS": plfs_wpr_cws,
    "UR usual": plfs_ur_usual,
    "UR CWS": plfs_ur_cws,
}.items():
    print(f"{label:16s}: {len(df)} rows")

## 9. Parse PLFS Table 4: employment structure

Table 4 is repeated for **male, female and person**. The third occurrence is the person block, which is the relevant version for the overall city comparison.

In [ ]:
plfs_table4_person = extract_nth_table(
    labour_txt,
    r"^Table 4:",
    occurrence=2,
    stop_patterns=[r"^Table 5:"],
)

plfs_table4_raw = parse_numeric_city_rows(
    plfs_table4_person,
    n_values=6,
    valid_cities=plfs_cities_raw,
)
plfs_table4_raw["city"] = plfs_table4_raw["city_raw"].map(clean_city_name)

plfs_employment_structure = plfs_table4_raw[
    ["city", "value_4", "value_5", "value_6"]
].rename(columns={
    "value_4": "regular_wage_person",
    "value_5": "casual_labour_person",
    "value_6": "all_employment_person",
})

# Table 4's "all" column is a 100% check.
plfs_employment_structure["self_employed_person"] = (
    plfs_table4_raw["value_4"] * 0  # placeholder overwritten below
)

# The table's third and fourth numeric values are:
# (4) all self-employed, (5) regular wage/salary, (6) casual labour, (7) all.
# Because n_values=6 includes (2) through (7), use value_3/value_4/value_5/value_6.
plfs_employment_structure = plfs_table4_raw[
    ["city", "value_3", "value_4", "value_5", "value_6"]
].rename(columns={
    "value_3": "self_employed_person",
    "value_4": "regular_wage_person",
    "value_5": "casual_labour_person",
    "value_6": "employment_total_check",
})

display(plfs_employment_structure.head())

## 10. Parse PLFS earnings tables

Table 7 provides average gross earnings from self-employment; Table 8 provides average monthly wage/salary from regular employment.

The values are for the relevant workers in CWS, not for every person in the city.

In [ ]:
def parse_plfs_earnings(table_start, next_table, prefix):
    block = extract_nth_table(
        labour_txt,
        table_start,
        occurrence=0,
        stop_patterns=[next_table],
    )
    raw = parse_numeric_city_rows(
        block,
        n_values=3,
        valid_cities=plfs_cities_raw,
    )
    raw["city"] = raw["city_raw"].map(clean_city_name)

    return raw[
        [
            "city",
            "value_1", "value_2", "value_3",
            "value_1_rse_star", "value_2_rse_star", "value_3_rse_star",
        ]
    ].rename(columns={
        "value_1": f"{prefix}_male",
        "value_2": f"{prefix}_female",
        "value_3": f"{prefix}_person",
        "value_1_rse_star": f"{prefix}_male_rse_star",
        "value_2_rse_star": f"{prefix}_female_rse_star",
        "value_3_rse_star": f"{prefix}_person_rse_star",
    })

plfs_self_earn = parse_plfs_earnings(
    r"^Table 7:", r"^Table 8:", "earn_self"
)
plfs_regular_earn = parse_plfs_earnings(
    r"^Table 8:", r"^Table 9:", "earn_regular"
)

display(
    plfs_regular_earn
    .sort_values("earn_regular_person", ascending=False)
    .head(10)
)

## 11. Parse PLFS Table 10: working hours

Table 10 is repeated by gender. The third occurrence is the person block. The fifth numeric column in the layout is the `all` hours figure, which becomes the overall city-level hours measure.

In [ ]:
plfs_table10_person = extract_nth_table(
    labour_txt,
    r"^Table 10:",
    occurrence=2,
    stop_patterns=[r"^Table 11:"],
)

plfs_hours_raw = parse_numeric_city_rows(
    plfs_table10_person,
    n_values=4,
    valid_cities=plfs_cities_raw,
)
plfs_hours_raw["city"] = plfs_hours_raw["city_raw"].map(clean_city_name)

plfs_hours = plfs_hours_raw[
    ["city", "value_4"]
].rename(columns={"value_4": "hours_person"})

display(plfs_hours.head())

## 12. Parse PLFS Table 12: youth NEET

Table 12 spans two pages/blocks. The first occurrence contains the 15–24 and 15–29 age groups.

We stop at the second Table 12 heading so the 30–59 / 15–59 block is not accidentally mixed into the 15–29 extraction.

In [ ]:
plfs_table12_first = extract_nth_table(
    labour_txt,
    r"^Table 12:",
    occurrence=0,
    stop_patterns=[r"^Table 12:", r"^Table 13\.1:"],
)

plfs_neet_raw = parse_numeric_city_rows(
    plfs_table12_first,
    n_values=6,
    valid_cities=plfs_cities_raw,
)
plfs_neet_raw["city"] = plfs_neet_raw["city_raw"].map(clean_city_name)

plfs_neet = plfs_neet_raw[
    ["city", "value_3", "value_6"]
].rename(columns={
    "value_3": "neet_15_24_person",
    "value_6": "neet_15_29_person",
})

display(
    plfs_neet.sort_values("neet_15_29_person", ascending=False).head(10)
)

## 13. Parse PLFS official RSE table

Table 13.1 reports the relative standard errors of LFPR, WPR and UR under usual status.

This gives a cleaner quality flag than relying only on the star markers embedded in the other tables.

In [ ]:
plfs_rse_block = extract_nth_table(
    labour_txt,
    r"^Table 13\.1:",
    occurrence=0,
    stop_patterns=[r"^Table 13\.2:"],
)

plfs_rse_raw = parse_numeric_city_rows(
    plfs_rse_block,
    n_values=9,
    valid_cities=plfs_cities_raw,
)
plfs_rse_raw["city"] = plfs_rse_raw["city_raw"].map(clean_city_name)

plfs_rse = plfs_rse_raw[
    [
        "city",
        "value_1", "value_2", "value_3",
        "value_4", "value_5", "value_6",
        "value_7", "value_8", "value_9",
    ]
].rename(columns={
    "value_1": "rse_lfpr_male_usual",
    "value_2": "rse_wpr_male_usual",
    "value_3": "rse_ur_male_usual",
    "value_4": "rse_lfpr_female_usual",
    "value_5": "rse_wpr_female_usual",
    "value_6": "rse_ur_female_usual",
    "value_7": "rse_lfpr_person_usual",
    "value_8": "rse_wpr_person_usual",
    "value_9": "rse_ur_person_usual",
})

display(plfs_rse.head())

## 14. Export the parsed PLFS source tables to CSV

These are the intermediate datasets produced directly from the PDF text extraction. They are useful for auditing and for reusing the extraction without rereading the PDF.

In [ ]:
plfs_exports = {
    "01_plfs_sample.csv": plfs_sample,
    "02_plfs_lfpr_usual.csv": plfs_lfpr_usual,
    "03_plfs_lfpr_cws.csv": plfs_lfpr_cws,
    "04_plfs_wpr_usual.csv": plfs_wpr_usual,
    "05_plfs_wpr_cws.csv": plfs_wpr_cws,
    "06_plfs_employment_structure.csv": plfs_employment_structure,
    "07_plfs_self_employment_earnings.csv": plfs_self_earn,
    "08_plfs_regular_wage_earnings.csv": plfs_regular_earn,
    "09_plfs_hours.csv": plfs_hours,
    "10_plfs_neet.csv": plfs_neet,
    "11_plfs_rse.csv": plfs_rse,
}

for filename, df in plfs_exports.items():
    df.to_csv(OUTPUT_DIR / filename, index=False)

print(f"{len(plfs_exports)} PLFS CSVs written.")

# ASUSE city-level extraction

## 15. Parse ASUSE Table 1: economic characteristics

The table contains establishments, workers, GVA per worker, GVA per establishment and emolument per hired worker.

The ASUSE report's GVA-per-worker measure is specific to its covered unincorporated non-agricultural economy.

In [ ]:
asuse_table1 = extract_nth_table(
    asuse_txt,
    r"^Table 1: Important Economic Characteristics for Million Plus Cities",
    occurrence=0,
    stop_patterns=[r"^Table 2:"],
)

asuse_econ_raw = parse_numeric_city_rows(
    asuse_table1,
    n_values=5,
    continuation_names={"Chhatrapati"},
)
asuse_econ_raw.loc[asuse_econ_raw["city_raw"] == "Chhatrapati", "city_raw"] = "Chhatrapati Sambhajinagar"
asuse_econ_raw["city"] = asuse_econ_raw["city_raw"].map(clean_city_name)

asuse_economic = asuse_econ_raw[
    ["city", "value_1", "value_2", "value_3", "value_4", "value_5"]
].rename(columns={
    "value_1": "asuse_establishments",
    "value_2": "asuse_workers",
    "value_3": "asuse_gva_per_worker",
    "value_4": "asuse_gva_per_establishment",
    "value_5": "asuse_emolument_per_hired_worker",
})

asuse_cities_raw = set(asuse_econ_raw["city_raw"])
asuse_cities = set(asuse_economic["city"])
print("ASUSE cities parsed:", len(asuse_cities))
display(asuse_economic.head(10))

## 16. Parse ASUSE Table 2: operational and employment characteristics

In [ ]:
asuse_table2 = extract_nth_table(
    asuse_txt,
    r"^Table 2: Important Operational and Employment Characteristics for Million Plus Cities",
    occurrence=0,
    stop_patterns=[r"^Annexure-IB:"],
)

asuse_operational_raw = parse_numeric_city_rows(
    asuse_table2,
    n_values=5,
    continuation_names={"Chhatrapati"},
)
asuse_operational_raw.loc[asuse_operational_raw["city_raw"] == "Chhatrapati", "city_raw"] = "Chhatrapati Sambhajinagar"
asuse_operational_raw["city"] = asuse_operational_raw["city_raw"].map(clean_city_name)

asuse_operational = asuse_operational_raw[
    ["city", "value_1", "value_2", "value_3", "value_4", "value_5"]
].rename(columns={
    "value_1": "proprietary_partnership_pct",
    "value_2": "hired_worker_establishment_pct",
    "value_3": "female_owned_proprietary_pct",
    "value_4": "hired_workers_pct",
    "value_5": "female_workers_pct",
})

display(asuse_operational.head(10))

## 17. Parse ASUSE Annexure-IB: RSEs

In [ ]:
asuse_rse_block = extract_nth_table(
    asuse_txt,
    r"^Annexure-IB: Relative Standard Error",
    occurrence=0,
    stop_patterns=[r"^Annexure-II"],
)

asuse_rse_raw = parse_numeric_city_rows(
    asuse_rse_block,
    n_values=6,
    continuation_names={"Chhatrapati"},
)
asuse_rse_raw.loc[asuse_rse_raw["city_raw"] == "Chhatrapati", "city_raw"] = "Chhatrapati Sambhajinagar"
asuse_rse_raw["city"] = asuse_rse_raw["city_raw"].map(clean_city_name)

asuse_rse = asuse_rse_raw[
    [
        "city",
        "value_1", "value_2", "value_3",
        "value_4", "value_5", "value_6",
    ]
].rename(columns={
    "value_1": "rse_establishments_pct",
    "value_2": "rse_workers_pct",
    "value_3": "rse_gva_per_worker_pct",
    "value_4": "rse_gva_per_establishment_pct",
    "value_5": "rse_emolument_per_hired_worker_pct",
    "value_6": "asuse_rse_sample_size",
})

display(asuse_rse.head())

## 18. Export the parsed ASUSE tables to CSV

In [ ]:
asuse_exports = {
    "12_asuse_economic.csv": asuse_economic,
    "13_asuse_operational.csv": asuse_operational,
    "14_asuse_rse.csv": asuse_rse,
}

for filename, df in asuse_exports.items():
    df.to_csv(OUTPUT_DIR / filename, index=False)

print(f"{len(asuse_exports)} ASUSE CSVs written.")

# PLFS July 2026

## 19. Parse the monthly urban labour-market pulse

The July bulletin is kept separate from the city cross-section.

We extract only **urban, persons aged 15+, person-level** values for March–July 2026 from Statements 1–3.

In [ ]:
def extract_statement_block(text, statement_number, next_statement_number):
    return extract_nth_table(
        text,
        rf"^Statement {statement_number}:",
        occurrence=0,
        stop_patterns=[rf"^Statement {next_statement_number}:"]
    )


def parse_monthly_urban_person(block, metric_name):
    """
    Extract the urban, person, age-15+ block from Statements 1-3.

    In the PDF text extraction, sector labels are vertically centred in their
    rowspan cells and can appear beside the middle month rather than before
    the first month. We therefore identify the second occurrence of the
    'age group: 15 years and above' block: rural is first, urban is second,
    rural+urban is third.
    """
    lines = block.splitlines()
    target = "age group: 15 years and above"
    starts = [i for i, line in enumerate(lines) if line.strip().lower() == target]

    if len(starts) < 3:
        raise ValueError(
            f"Expected at least 3 '{target}' blocks, found {len(starts)}."
        )

    start = starts[1]
    end = len(lines)
    for j in range(start + 1, len(lines)):
        if lines[j].strip().lower() == "age group: all age":
            end = j
            break

    month_re = re.compile(
        r"^(March|April|May|June|July),\s*2026\s+"
        r"([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s*$"
    )

    rows = []
    for raw_line in lines[start + 1:end]:
        line = raw_line.strip()
        m = month_re.match(line)
        if m:
            rows.append({
                "month": m.group(1),
                "year": 2026,
                "metric": metric_name,
                "sector": "urban",
                "age_group": "15 years and above",
                "male": float(m.group(2)),
                "female": float(m.group(3)),
                "person": float(m.group(4)),
            })

    out = pd.DataFrame(rows)
    expected_months = {"March", "April", "May", "June", "July"}
    assert set(out["month"]) == expected_months
    assert len(out) == 5
    return out


statement1 = extract_statement_block(july_txt, 1, 2)
statement2 = extract_statement_block(july_txt, 2, 3)
statement3 = extract_statement_block(july_txt, 3, 4)

july_lfpr = parse_monthly_urban_person(statement1, "LFPR")
july_wpr = parse_monthly_urban_person(statement2, "WPR")
july_ur = parse_monthly_urban_person(statement3, "UR")

month_order = ["March", "April", "May", "June", "July"]
plfs_urban_monthly = pd.concat(
    [july_lfpr, july_wpr, july_ur],
    ignore_index=True,
)
plfs_urban_monthly["month"] = pd.Categorical(
    plfs_urban_monthly["month"],
    categories=month_order,
    ordered=True,
)
plfs_urban_monthly = (
    plfs_urban_monthly
    .sort_values(["metric", "month"])
    .reset_index(drop=True)
)

print("Rows:", len(plfs_urban_monthly))
display(plfs_urban_monthly)


## 20. Save the July PLFS extraction

In [ ]:
plfs_urban_monthly.to_csv(
    OUTPUT_DIR / "15_plfs_urban_monthly_march_july_2026.csv",
    index=False
)

# Quality control before analysis

## 21. Coverage, duplicates and range checks

The notebook stops before analysis if the extracted city universe is not the expected 46 cities or if core percentage fields fall outside 0–100.

In [ ]:
def basic_qc(df):
    return {
        "rows": len(df),
        "unique_cities": df["city"].nunique(),
        "duplicate_city_rows": int(df["city"].duplicated().sum()),
        "missing_city": int(df["city"].isna().sum()),
    }

qc = pd.DataFrame(
    [{"table": name, **basic_qc(df)} for name, df in {
        "PLFS sample": plfs_sample,
        "PLFS LFPR usual": plfs_lfpr_usual,
        "PLFS WPR usual": plfs_wpr_usual,
        "PLFS employment": plfs_employment_structure,
        "PLFS earnings": plfs_regular_earn,
        "PLFS NEET": plfs_neet,
        "PLFS RSE": plfs_rse,
        "ASUSE economic": asuse_economic,
        "ASUSE operational": asuse_operational,
        "ASUSE RSE": asuse_rse,
    }.items()]
)

display(qc)

range_qc = pd.DataFrame([
    {
        "check": "PLFS LFPR usual person in 0-100",
        "passed": plfs_lfpr_usual["lfpr_usual_person"].between(0, 100).all(),
    },
    {
        "check": "PLFS WPR usual person in 0-100",
        "passed": plfs_wpr_usual["wpr_usual_person"].between(0, 100).all(),
    },
    {
        "check": "PLFS UR usual person in 0-100",
        "passed": plfs_ur_usual["ur_usual_person"].between(0, 100).all(),
    },
    {
        "check": "PLFS employment structure sums to ~100",
        "passed": np.allclose(
            plfs_employment_structure["self_employed_person"]
            + plfs_employment_structure["regular_wage_person"]
            + plfs_employment_structure["casual_labour_person"],
            100,
            atol=0.2,
        ),
    },
    {
        "check": "ASUSE hired workers percentage in 0-100",
        "passed": asuse_operational["hired_workers_pct"].between(0, 100).all(),
    },
    {
        "check": "ASUSE female workers percentage in 0-100",
        "passed": asuse_operational["female_workers_pct"].between(0, 100).all(),
    },
    {
        "check": "July urban monthly extraction has 15 rows",
        "passed": len(plfs_urban_monthly) == 15,
    },
])

range_qc["passed"] = range_qc["passed"].astype(bool)
display(range_qc)

qc.to_csv(OUTPUT_DIR / "16_quality_checks.csv", index=False)
range_qc.to_csv(OUTPUT_DIR / "17_range_checks.csv", index=False)


## 22A. Row-level extraction audit

A parser can technically produce numeric rows while still duplicating a table or swallowing a continuation page. This audit checks the expected one-row-per-city structure explicitly.


In [ ]:
city_tables = {
    "PLFS LFPR usual": plfs_lfpr_usual,
    "PLFS LFPR CWS": plfs_lfpr_cws,
    "PLFS WPR usual": plfs_wpr_usual,
    "PLFS WPR CWS": plfs_wpr_cws,
    "PLFS UR usual": plfs_ur_usual,
    "PLFS UR CWS": plfs_ur_cws,
    "PLFS employment": plfs_employment_structure,
    "PLFS self earnings": plfs_self_earn,
    "PLFS regular earnings": plfs_regular_earn,
    "PLFS hours": plfs_hours,
    "PLFS NEET": plfs_neet,
    "PLFS RSE": plfs_rse,
    "ASUSE economic": asuse_economic,
    "ASUSE operational": asuse_operational,
    "ASUSE RSE": asuse_rse,
}

row_audit = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "unique_city_keys": df["city"].nunique(),
        "one_row_per_city": (
            len(df) == 46
            and df["city"].nunique() == 46
            and not df["city"].duplicated().any()
        ),
    }
    for name, df in city_tables.items()
])

display(row_audit)

assert row_audit["one_row_per_city"].all(), (
    "At least one source table is not one-row-per-city."
)
assert len(plfs_cities) == 46
assert len(asuse_cities) == 46
assert plfs_cities == asuse_cities
assert range_qc["passed"].all()

row_audit.to_csv(OUTPUT_DIR / "18_row_level_extraction_audit.csv", index=False)


## 22. Confirm that PLFS and ASUSE city names match

This explicitly checks the merge universe rather than assuming the datasets use identical labels.

In [ ]:
plfs_only = sorted(plfs_cities - asuse_cities)
asuse_only = sorted(asuse_cities - plfs_cities)

print("Cities in PLFS but not ASUSE:", plfs_only)
print("Cities in ASUSE but not PLFS:", asuse_only)

assert not plfs_only and not asuse_only


# Merge and derive the final analytical dataset

## 23. Merge all selected PLFS and ASUSE tables

The merge is intentionally performed **after** source-specific extraction and QC.

In [ ]:
master = (
    plfs_sample
    .merge(
        plfs_lfpr_usual[[
            "city", "lfpr_usual_male", "lfpr_usual_female", "lfpr_usual_person"
        ]],
        on="city", how="inner"
    )
    .merge(
        plfs_lfpr_cws[[
            "city", "lfpr_cws_male", "lfpr_cws_female", "lfpr_cws_person"
        ]],
        on="city", how="inner"
    )
    .merge(
        plfs_wpr_usual[[
            "city", "wpr_usual_male", "wpr_usual_female", "wpr_usual_person"
        ]],
        on="city", how="inner"
    )
    .merge(
        plfs_wpr_cws[[
            "city", "wpr_cws_male", "wpr_cws_female", "wpr_cws_person"
        ]],
        on="city", how="inner"
    )
    .merge(
        plfs_ur_usual[[
            "city", "ur_usual_male", "ur_usual_female", "ur_usual_person"
        ]],
        on="city", how="inner"
    )
    .merge(
        plfs_ur_cws[[
            "city", "ur_cws_male", "ur_cws_female", "ur_cws_person"
        ]],
        on="city", how="inner"
    )
    .merge(plfs_employment_structure, on="city", how="inner")
    .merge(plfs_self_earn[["city", "earn_self_person"]], on="city", how="inner")
    .merge(plfs_regular_earn[["city", "earn_regular_person"]], on="city", how="inner")
    .merge(plfs_hours, on="city", how="inner")
    .merge(plfs_neet, on="city", how="inner")
    .merge(
        plfs_rse[[
            "city",
            "rse_lfpr_person_usual",
            "rse_wpr_person_usual",
            "rse_ur_person_usual",
        ]],
        on="city", how="inner"
    )
    .merge(asuse_economic, on="city", how="inner")
    .merge(asuse_operational, on="city", how="inner")
    .merge(
        asuse_rse[[
            "city",
            "rse_gva_per_worker_pct",
            "rse_gva_per_establishment_pct",
            "rse_emolument_per_hired_worker_pct",
        ]],
        on="city", how="inner"
    )
)

assert len(master) == 46
print("Merged shape:", master.shape)
display(master.head())

## 24. Create transparent derived variables

These are calculations made from the extracted source fields. They are not additional official estimates.

In [ ]:
master["lfpr_gender_gap_pp"] = (
    master["lfpr_usual_male"] - master["lfpr_usual_female"]
)

master["wpr_gender_gap_pp"] = (
    master["wpr_usual_male"] - master["wpr_usual_female"]
)

master["non_regular_employment_pct"] = (
    master["self_employed_person"] + master["casual_labour_person"]
)

master["workers_per_establishment"] = (
    master["asuse_workers"] / master["asuse_establishments"]
)

master["employment_structure_check"] = (
    master["self_employed_person"]
    + master["regular_wage_person"]
    + master["casual_labour_person"]
)

master["plfs_key_indicator_high_rse"] = (
    master[
        ["rse_lfpr_person_usual", "rse_wpr_person_usual", "rse_ur_person_usual"]
    ].max(axis=1) >= 40
)

master["asuse_gva_per_worker_high_rse"] = (
    master["rse_gva_per_worker_pct"] >= 40
)

master["gva_per_worker_thousand_rupees"] = (
    master["asuse_gva_per_worker"] / 1000
)

display(
    master[[
        "city",
        "lfpr_usual_person",
        "wpr_usual_person",
        "ur_usual_person",
        "regular_wage_person",
        "asuse_gva_per_worker",
        "asuse_workers",
        "lfpr_gender_gap_pp",
    ]].head()
)

## 25. Save the main 46-city master CSV

In [ ]:
master.to_csv(OUTPUT_DIR / "18_million_plus_city_master.csv", index=False)
print("Saved:", OUTPUT_DIR / "18_million_plus_city_master.csv")

# Analysis tables

## 26. Scale of the unincorporated business economy

Use this as a size table, not a productivity ranking.

In [ ]:
top_scale = (
    master[[
        "city",
        "asuse_establishments",
        "asuse_workers",
        "workers_per_establishment",
    ]]
    .sort_values("asuse_workers", ascending=False)
    .head(10)
)

display(top_scale)
top_scale.to_csv(OUTPUT_DIR / "19_top10_asuse_scale.csv", index=False)

## 27. Productivity: ASUSE GVA per worker

Show the highest and lowest observations, with RSE beside the estimate.

In [ ]:
gva_rank = (
    master[[
        "city",
        "asuse_gva_per_worker",
        "rse_gva_per_worker_pct",
        "asuse_workers",
    ]]
    .sort_values("asuse_gva_per_worker", ascending=False)
)

display(gva_rank.head(10))
display(gva_rank.tail(10).sort_values("asuse_gva_per_worker"))

gva_rank.to_csv(OUTPUT_DIR / "20_gva_per_worker_all_cities.csv", index=False)

## 28. Labour utilisation vs enterprise productivity

This is the main cross-dataset test:

**PLFS WPR (usual status) ↔ ASUSE GVA per worker**

The correlation is a descriptive cross-sectional statistic, not a causal estimate.

In [ ]:
wpr_gva = master[[
    "city",
    "wpr_usual_person",
    "asuse_gva_per_worker",
    "asuse_workers",
    "rse_gva_per_worker_pct",
]].dropna()

print(
    "Pearson correlation:",
    round(
        wpr_gva["wpr_usual_person"].corr(wpr_gva["asuse_gva_per_worker"]),
        3
    )
)

display(wpr_gva.sort_values("wpr_usual_person", ascending=False).head(10))
wpr_gva.to_csv(OUTPUT_DIR / "21_wpr_vs_gva_per_worker.csv", index=False)

## 29. Employment structure

Sort by regular wage/salary share. This is a direct use of PLFS Table 4.

In [ ]:
employment_structure_table = (
    master[[
        "city",
        "self_employed_person",
        "regular_wage_person",
        "casual_labour_person",
        "non_regular_employment_pct",
    ]]
    .sort_values("regular_wage_person", ascending=False)
)

display(employment_structure_table.head(15))
employment_structure_table.to_csv(
    OUTPUT_DIR / "22_employment_structure_all_cities.csv",
    index=False
)

## 30. Gender participation gaps

In [ ]:
gender_gap_table = (
    master[[
        "city",
        "lfpr_usual_male",
        "lfpr_usual_female",
        "lfpr_gender_gap_pp",
        "rse_lfpr_person_usual",
    ]]
    .sort_values("lfpr_gender_gap_pp", ascending=False)
)

display(gender_gap_table.head(15))
gender_gap_table.to_csv(
    OUTPUT_DIR / "23_gender_lfpr_gap_all_cities.csv",
    index=False
)

## 31. Youth NEET

In [ ]:
neet_table = (
    master[[
        "city",
        "neet_15_24_person",
        "neet_15_29_person",
    ]]
    .sort_values("neet_15_29_person", ascending=False)
)

display(neet_table.head(15))
neet_table.to_csv(OUTPUT_DIR / "24_neet_15_29_all_cities.csv", index=False)

# Charts and graphs

The chart set is intentionally selective. Each chart answers a different part of the story rather than displaying every variable.

## 32. Chart 1 — Urban LFPR and WPR, March–July 2026

LFPR and WPR sit on a comparable scale, so a two-line time series is useful for the current urban pulse.

In [ ]:
trend = (
    plfs_urban_monthly[
        plfs_urban_monthly["metric"].isin(["LFPR", "WPR"])
    ]
    .pivot(index="month", columns="metric", values="person")
    .reindex(month_order)
)

fig, ax = plt.subplots(figsize=(10, 6))
for metric in ["LFPR", "WPR"]:
    ax.plot(trend.index, trend[metric], marker="o", label=metric)

ax.set_title("Urban LFPR and WPR, March–July 2026")
ax.set_xlabel("Survey month")
ax.set_ylabel("Per cent")
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()

chart_path = CHART_DIR / "01_urban_lfpr_wpr_trend.png"
fig.savefig(chart_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

## 33. Chart 2 — Urban unemployment rate, March–July 2026

UR gets its own chart because its percentage range is materially smaller than LFPR/WPR.

In [ ]:
ur_trend = (
    plfs_urban_monthly[plfs_urban_monthly["metric"] == "UR"]
    .set_index("month")
    .reindex(month_order)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ur_trend.index, ur_trend["person"], marker="o")

ax.set_title("Urban unemployment rate, March–July 2026")
ax.set_xlabel("Survey month")
ax.set_ylabel("Unemployment rate (%)")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

chart_path = CHART_DIR / "02_urban_ur_trend.png"
fig.savefig(chart_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

## 34. Chart 3 — WPR vs ASUSE GVA per worker

This is the main PLFS–ASUSE cross-sectional graphic.

Bubble size is based on ASUSE estimated workers; only a small set of extreme observations are labelled so the graphic remains readable.

In [ ]:
plot_df = master[[
    "city",
    "wpr_usual_person",
    "asuse_gva_per_worker",
    "asuse_workers",
]].dropna()

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(
    plot_df["wpr_usual_person"],
    plot_df["asuse_gva_per_worker"],
    s=plot_df["asuse_workers"] / 900,
    alpha=0.65,
    edgecolor="black",
    linewidth=0.4,
)

ax.set_title("Labour utilisation and unincorporated-enterprise productivity")
ax.set_xlabel("PLFS WPR, usual status (%)")
ax.set_ylabel("ASUSE GVA per worker (₹)")
ax.grid(alpha=0.2)

label_rows = pd.concat([
    plot_df.nlargest(4, "asuse_gva_per_worker"),
    plot_df.nlargest(4, "wpr_usual_person"),
]).drop_duplicates("city")

for _, row in label_rows.iterrows():
    ax.annotate(
        row["city"],
        (row["wpr_usual_person"], row["asuse_gva_per_worker"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

fig.tight_layout()
chart_path = CHART_DIR / "03_wpr_vs_gva_per_worker.png"
fig.savefig(chart_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

## 35. Chart 4 — Largest male–female LFPR gaps

A horizontal bar chart avoids the long-city-name problem and makes the percentage-point gap directly readable.

In [ ]:
gap_plot = (
    master[[
        "city",
        "lfpr_gender_gap_pp",
    ]]
    .sort_values("lfpr_gender_gap_pp", ascending=False)
    .head(12)
    .sort_values("lfpr_gender_gap_pp")
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(gap_plot["city"], gap_plot["lfpr_gender_gap_pp"])

ax.set_title("Largest male–female LFPR gaps")
ax.set_xlabel("Male LFPR minus female LFPR (percentage points)")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()

chart_path = CHART_DIR / "04_gender_lfpr_gap.png"
fig.savefig(chart_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

## 36. Chart 5 — Employment structure in regular-wage-intensive cities

A stacked horizontal bar is appropriate because the three PLFS employment-status categories form the distribution of workers.

In [ ]:
structure_plot = (
    master[[
        "city",
        "self_employed_person",
        "regular_wage_person",
        "casual_labour_person",
    ]]
    .sort_values("regular_wage_person", ascending=True)
    .tail(10)
)

fig, ax = plt.subplots(figsize=(11, 7))

left = np.zeros(len(structure_plot))

for col, label in [
    ("self_employed_person", "Self-employed"),
    ("regular_wage_person", "Regular wage/salary"),
    ("casual_labour_person", "Casual labour"),
]:
    ax.barh(
        structure_plot["city"],
        structure_plot[col],
        left=left,
        label=label,
    )
    left = left + structure_plot[col].to_numpy()

ax.set_title("Employment structure in cities with the highest regular-wage share")
ax.set_xlabel("Share of workers (%)")
ax.set_xlim(0, 100)
ax.grid(axis="x", alpha=0.2)
ax.legend(loc="lower right")

fig.tight_layout()
chart_path = CHART_DIR / "05_employment_structure_regular_wage_intensive.png"
fig.savefig(chart_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

## 37. Chart 6 — ASUSE GVA per worker: 10 highest and 10 lowest

A horizontal ranking is readable for these long city names. RSE remains in the supporting table, so the chart should be read alongside the quality information.

In [ ]:
gva_extremes = pd.concat([
    master.nlargest(10, "asuse_gva_per_worker"),
    master.nsmallest(10, "asuse_gva_per_worker"),
]).drop_duplicates("city").sort_values("asuse_gva_per_worker")

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(gva_extremes["city"], gva_extremes["asuse_gva_per_worker"])

ax.set_title("ASUSE GVA per worker: 10 highest and 10 lowest cities")
ax.set_xlabel("GVA per worker (₹)")
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()
chart_path = CHART_DIR / "06_gva_per_worker_extremes.png"
fig.savefig(chart_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

# Final analysis exports

## 38. Build the compact newsroom/story table

This is the most useful all-purpose output for Excel, data visualisation or further reporting.

In [ ]:
story_columns = [
    "city",
    "persons_surveyed",
    "lfpr_usual_person",
    "wpr_usual_person",
    "ur_usual_person",
    "lfpr_usual_male",
    "lfpr_usual_female",
    "lfpr_gender_gap_pp",
    "self_employed_person",
    "regular_wage_person",
    "casual_labour_person",
    "earn_self_person",
    "earn_regular_person",
    "hours_person",
    "neet_15_29_person",
    "asuse_establishments",
    "asuse_workers",
    "workers_per_establishment",
    "asuse_gva_per_worker",
    "asuse_gva_per_establishment",
    "asuse_emolument_per_hired_worker",
    "hired_worker_establishment_pct",
    "hired_workers_pct",
    "female_owned_proprietary_pct",
    "female_workers_pct",
    "rse_gva_per_worker_pct",
    "plfs_key_indicator_high_rse",
    "asuse_gva_per_worker_high_rse",
]

story_table = master[story_columns].copy()
story_table.to_csv(
    OUTPUT_DIR / "25_newsroom_story_table.csv",
    index=False
)

display(story_table.head())

## 39. Data dictionary

In [ ]:
data_dictionary = pd.DataFrame([
    ["lfpr_usual_*", "PLFS", "Table 2", "LFPR by gender/person, usual status, age 15+"],
    ["lfpr_cws_*", "PLFS", "Table 2.1", "LFPR by gender/person, CWS, age 15+"],
    ["wpr_usual_*", "PLFS", "Table 3", "WPR by gender/person, usual status, age 15+"],
    ["wpr_cws_*", "PLFS", "Table 3.1", "WPR by gender/person, CWS, age 15+"],
    ["self_employed_person", "PLFS", "Table 4", "Share of workers who are self-employed"],
    ["regular_wage_person", "PLFS", "Table 4", "Share of workers in regular wage/salary employment"],
    ["casual_labour_person", "PLFS", "Table 4", "Share of workers in casual labour"],
    ["earn_self_person", "PLFS", "Table 7", "Average gross earnings from self-employment, CWS"],
    ["earn_regular_person", "PLFS", "Table 8", "Average monthly wage/salary from regular employment, CWS"],
    ["hours_person", "PLFS", "Table 10", "Average hours actually worked per week"],
    ["ur_usual_*", "PLFS", "Table 11", "UR by gender/person, usual status"],
    ["ur_cws_*", "PLFS", "Table 11.1", "UR by gender/person, CWS"],
    ["neet_15_29_person", "PLFS", "Table 12", "NEET, age 15–29, person"],
    ["rse_lfpr_person_usual", "PLFS", "Table 13.1", "RSE of person LFPR, usual status"],
    ["rse_wpr_person_usual", "PLFS", "Table 13.1", "RSE of person WPR, usual status"],
    ["rse_ur_person_usual", "PLFS", "Table 13.1", "RSE of person UR, usual status"],
    ["asuse_establishments", "ASUSE", "Table 1", "Estimated number of establishments"],
    ["asuse_workers", "ASUSE", "Table 1", "Estimated number of workers"],
    ["asuse_gva_per_worker", "ASUSE", "Table 1", "GVA per worker (₹)"],
    ["asuse_gva_per_establishment", "ASUSE", "Table 1", "GVA per establishment (₹)"],
    ["asuse_emolument_per_hired_worker", "ASUSE", "Table 1", "Emolument per hired worker (₹)"],
    ["proprietary_partnership_pct", "ASUSE", "Table 2", "Proprietary and partnership establishments (%)"],
    ["hired_worker_establishment_pct", "ASUSE", "Table 2", "Hired-worker establishments (%)"],
    ["female_owned_proprietary_pct", "ASUSE", "Table 2", "Female-owned proprietary establishments (%)"],
    ["hired_workers_pct", "ASUSE", "Table 2", "Hired workers as share of workers (%)"],
    ["female_workers_pct", "ASUSE", "Table 2", "Female workers as share of workers (%)"],
    ["rse_gva_per_worker_pct", "ASUSE", "Annexure-IB", "RSE of GVA per worker (%)"],
], columns=["field", "source", "table", "definition"])

data_dictionary.to_csv(
    OUTPUT_DIR / "26_data_dictionary.csv",
    index=False
)

display(data_dictionary.head(15))

## 40. Editorial-quality flags

These flags are intended to keep uncertainty visible during reporting.

They do **not** rank or score cities; they simply mark observations where the source-reported RSE warrants additional caution.

In [ ]:
editorial_flags = master[[
    "city",
    "wpr_usual_person",
    "asuse_gva_per_worker",
    "regular_wage_person",
    "lfpr_gender_gap_pp",
    "neet_15_29_person",
    "rse_gva_per_worker_pct",
    "plfs_key_indicator_high_rse",
    "asuse_gva_per_worker_high_rse",
]].copy()

editorial_flags["plfs_rse_caution"] = editorial_flags["plfs_key_indicator_high_rse"]
editorial_flags["asuse_gva_rse_caution"] = editorial_flags["asuse_gva_per_worker_high_rse"]

editorial_flags.to_csv(
    OUTPUT_DIR / "27_editorial_flags.csv",
    index=False
)

display(editorial_flags.head())

## 41. Final file inventory

The finished workflow should leave behind both the intermediate extraction products and the final analytical products.

In [ ]:
inventory = []

for folder in [RAW_TEXT_DIR, OUTPUT_DIR, CHART_DIR]:
    for path in sorted(folder.iterdir()):
        if path.is_file():
            inventory.append({
                "folder": folder.name,
                "file": path.name,
                "size_kb": round(path.stat().st_size / 1024, 1),
            })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)

print("\nEND-TO-END WORKFLOW COMPLETE")